# Unduh & Verifikasi Dataset SmSA (IndoNLU)
### **Mata Kuliah:** Pembelajaran Mendalam (IF25-40401) — Studio Proyek 3
### **Tugas:** Sentiment Analysis Prosa (SmSA) — Klasifikasi Sentimen 3 Kelas

Notebook ini bertujuan untuk mengunduh dataset resmi **SmSA** (*Sentence-level Sentiment Analysis*) dari repositori benchmark **IndoNLP/indonlu** secara deterministik dan *reproducible*.

#### **Jaminan Reproducibility & Integritas Data:**
1. **Pinned Commit Revision:** URL unduhan dikunci pada commit SHA spesifik (`ce728f6926a36174b9923dfe49d6a6839b6e9bb7`) daripada branch `master` yang bersifat dinamis. Ini menjamin data yang digunakan selalu identik di seluruh sesi eksperimen (Google Colab, Kaggle, maupun lokal).
2. **Verifikasi Checksum (SHA-256):** Setiap berkas yang diunduh langsung divalidasi keutuhannya terhadap hash kriptografis resmi untuk mendeteksi *corrupted download* atau *payload mismatch*.
3. **Inspeksi Awal Distribusi:** Memuat data ke Pandas DataFrame untuk memverifikasi jumlah baris, integritas kolom (`text` dan `sentiment`), serta distribusi kelas (*positive*, *neutral*, *negative*).

--- 
## 1. Konfigurasi Parameter & URL Dataset

In [1]:
import os
import hashlib
import urllib.request
import pandas as pd

# 1. Definisi Commit Pinned & Sumber Repositori
GITHUB_REPO = "IndoNLP/indonlu"
COMMIT_HASH = "ce728f6926a36174b9923dfe49d6a6839b6e9bb7"
BASE_RAW_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{COMMIT_HASH}/dataset/smsa_doc-sentiment-prosa"

# 2. Direktori Penyimpanan Lokal
DATA_DIR = "data/smsa"
os.makedirs(DATA_DIR, exist_ok=True)

# 3. Metadata Berkas: Nama File, URL, Ekspektasi SHA-256, dan Jumlah Baris
DATASET_METADATA = {
    "train": {
        "filename": "train_preprocess.tsv",
        "url": f"{BASE_RAW_URL}/train_preprocess.tsv",
        "expected_sha256": "50f38ceed9b31521bf1581e126620532cc9b790712938159a2cdcf6906977a9b",
        "expected_rows": 11000
    },
    "valid": {
        "filename": "valid_preprocess.tsv",
        "url": f"{BASE_RAW_URL}/valid_preprocess.tsv",
        "expected_sha256": "6ab41ddc9d58a35086f05ebd2e209c74cb03d87d4f51d6abdfba674eafbefa74",
        "expected_rows": 1260
    },
    "test": {
        "filename": "test_preprocess.tsv",
        "url": f"{BASE_RAW_URL}/test_preprocess.tsv",
        "expected_sha256": "4e8016daa8e4e1b193f2a28a79d3a7804dd4e3f926a3ab7e8a64264f5f416a6f",
        "expected_rows": 500
    }
}

print(f"Konfigurasi siap.")
print(f"Base URL Pinned Commit: {BASE_RAW_URL}")
print(f"Target direktori: {os.path.abspath(DATA_DIR)}")

Konfigurasi siap.
Base URL Pinned Commit: https://raw.githubusercontent.com/IndoNLP/indonlu/ce728f6926a36174b9923dfe49d6a6839b6e9bb7/dataset/smsa_doc-sentiment-prosa
Target direktori: /Users/glutto/Documents/InformaticsEngineering/DeepLearning/finetuning-pretrained-transformer/data/smsa


--- 
## 2. Fungsi Pengunduhan & Validasi SHA-256

In [2]:
def calculate_sha256(filepath: str) -> str:
    """Menghitung hash SHA-256 dari berkas biner lokal."""
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(65536), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

def download_and_verify(split_name: str, meta: dict, target_dir: str) -> str:
    """Mengunduh berkas dataset jika belum ada dan memverifikasi integritas checksum SHA-256."""
    dest_path = os.path.join(target_dir, meta["filename"])
    url = meta["url"]
    expected_hash = meta["expected_sha256"]
    
    # Cek apakah berkas sudah ada dan valid
    if os.path.exists(dest_path):
        current_hash = calculate_sha256(dest_path)
        if current_hash == expected_hash:
            print(f"[CACHE HIT] {meta['filename']} sudah ada dan valid (SHA-256 cocok).")
            return dest_path
        else:
            print(f"[WARNING] {meta['filename']} ditemukan tetapi hash tidak cocok. Mengunduh ulang...")
            os.remove(dest_path)

    # Pengunduhan berkas
    print(f"[DOWNLOADING] Mengunduh {split_name} dari {url}...")
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    )
    with urllib.request.urlopen(req) as response, open(dest_path, "wb") as out_file:
        out_file.write(response.read())

    # Verifikasi Integritas SHA-256
    downloaded_hash = calculate_sha256(dest_path)
    if downloaded_hash != expected_hash:
        raise ValueError(
            f"Integritas data korup untuk {meta['filename']}!\n"
            f"Expected SHA-256: {expected_hash}\n"
            f"Actual SHA-256:   {downloaded_hash}"
        )
    
    file_size_kb = os.path.getsize(dest_path) / 1024
    print(f"[SUCCESS] {meta['filename']} berhasil diunduh ({file_size_kb:.1f} KB). SHA-256 terverifikasi valid!\n")
    return dest_path

# Eksekusi pengunduhan seluruh split
downloaded_files = {}
for split, meta in DATASET_METADATA.items():
    downloaded_files[split] = download_and_verify(split, meta, DATA_DIR)

[DOWNLOADING] Mengunduh train dari https://raw.githubusercontent.com/IndoNLP/indonlu/ce728f6926a36174b9923dfe49d6a6839b6e9bb7/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv...
[SUCCESS] train_preprocess.tsv berhasil diunduh (2135.5 KB). SHA-256 terverifikasi valid!

[DOWNLOADING] Mengunduh valid dari https://raw.githubusercontent.com/IndoNLP/indonlu/ce728f6926a36174b9923dfe49d6a6839b6e9bb7/dataset/smsa_doc-sentiment-prosa/valid_preprocess.tsv...
[SUCCESS] valid_preprocess.tsv berhasil diunduh (241.2 KB). SHA-256 terverifikasi valid!

[DOWNLOADING] Mengunduh test dari https://raw.githubusercontent.com/IndoNLP/indonlu/ce728f6926a36174b9923dfe49d6a6839b6e9bb7/dataset/smsa_doc-sentiment-prosa/test_preprocess.tsv...
[SUCCESS] test_preprocess.tsv berhasil diunduh (74.2 KB). SHA-256 terverifikasi valid!



--- 
## 3. Pemuatan & Validasi Struktur Data (Pandas)

In [3]:
dataframes = {}

for split, meta in DATASET_METADATA.items():
    filepath = downloaded_files[split]
    
    # SmSA format: TSV tanpa header -> Kolom 0: text, Kolom 1: sentiment
    df = pd.read_csv(
        filepath,
        sep="\t",
        header=None,
        names=["text", "sentiment"],
        quoting=3  # QUOTE_NONE untuk menghindari isu tanda petik di teks ulasan
    )
    dataframes[split] = df
    
    # Verifikasi jumlah baris
    actual_rows = len(df)
    expected_rows = meta["expected_rows"]
    assert actual_rows == expected_rows, f"Mismatch baris {split}: dapat {actual_rows}, harap {expected_rows}!"
    
    print(f"Split: '{split.upper()}' | Total Baris: {len(df):,} | Missing Values: {df.isna().sum().to_dict()}")

Split: 'TRAIN' | Total Baris: 11,000 | Missing Values: {'text': 0, 'sentiment': 0}
Split: 'VALID' | Total Baris: 1,260 | Missing Values: {'text': 0, 'sentiment': 0}
Split: 'TEST' | Total Baris: 500 | Missing Values: {'text': 0, 'sentiment': 0}


--- 
## 4. Analisis Distribusi Kelas Sentimen
Memeriksa keseimbangan kelas (`positive`, `neutral`, `negative`) pada masing-masing split (*train*, *valid*, *test*).

In [4]:
distribution_summary = []

for split, df in dataframes.items():
    counts = df["sentiment"].value_counts()
    percentages = df["sentiment"].value_counts(normalize=True) * 100
    for label in ["positive", "neutral", "negative"]:
        distribution_summary.append({
            "Split": split.capitalize(),
            "Label": label,
            "Jumlah Sampel": counts.get(label, 0),
            "Proporsi (%)": round(percentages.get(label, 0.0), 2)
        })

df_dist = pd.DataFrame(distribution_summary)
display(df_dist)

,Split,Label,Jumlah Sampel,Proporsi (%)
0,Train,positive,6416,58.33
1,Train,neutral,1148,10.44
2,Train,negative,3436,31.24
3,Valid,positive,735,58.33
4,Valid,neutral,131,10.40
5,Valid,negative,394,31.27
6,Test,positive,208,41.60
7,Test,neutral,88,17.60
8,Test,negative,204,40.80


--- 
## 5. Pratinjau Sampel Data SmSA

In [5]:
print("=== 5 Contoh Sampel Data Latih (Train) ===")
for idx, row in dataframes["train"].head(5).iterrows():
    print(f"[{idx+1}] Label: {row['sentiment'].upper()}")
    print(f"    Teks : {row['text']}\n")

=== 5 Contoh Sampel Data Latih (Train) ===
[1] Label: POSITIVE
    Teks : warung ini dimiliki oleh pengusaha pabrik tahu yang sudah puluhan tahun terkenal membuat tahu putih di bandung . tahu berkualitas , dipadu keahlian memasak , dipadu kretivitas , jadilah warung yang menyajikan menu utama berbahan tahu , ditambah menu umum lain seperti ayam . semuanya selera indonesia . harga cukup terjangkau . jangan lewatkan tahu bletoka nya , tidak kalah dengan yang asli dari tegal !

[2] Label: NEUTRAL
    Teks : mohon ulama lurus dan k212 mmbri hujjah partai apa yang harus diwlh agar suara islam tidak pecah-pecah

[3] Label: POSITIVE
    Teks : lokasi strategis di jalan sumatera bandung . tempat nya nyaman terutama sofa di lantai 2 . paella nya enak , sangat pas dimakan dengan minum bir dingin . appetiser nya juga enak-enak .

[4] Label: POSITIVE
    Teks : betapa bahagia nya diri ini saat unboxing paket dan barang nya bagus ! menetapkan beli lagi !

[5] Label: NEGATIVE
    Teks : duh . jadi m

--- 
## 6. Pemeriksaan Integritas Lanjutan (Sesuai Spesifikasi Tugas)
Spesifikasi Tugas Bagian C & Rekomendasi Konfigurasi mewajibkan pengecekan:
1. Duplikasi baris teks.
2. Kontradiksi label (teks identik dengan label berbeda).
3. Potensi kebocoran data validasi/tes ke data latih (*data leakage*).

In [6]:
df_train = dataframes["train"]
df_valid = dataframes["valid"]
df_test = dataframes["test"]

# 1. Deteksi duplikat teks di dalam set latih
n_duplicates = df_train.duplicated(subset=["text"]).sum()
print(f"Jumlah teks duplikat di data latih: {n_duplicates}")

# 2. Deteksi teks berkontradiksi (teks sama label berbeda)
text_label_nunique = df_train.groupby("text")["sentiment"].nunique()
contradictory_texts = text_label_nunique[text_label_nunique > 1]
print(f"Jumlah teks dengan label berkontradiksi di data latih: {len(contradictory_texts)}")

# 3. Deteksi kebocoran data (overlap antara train dan valid/test)
train_texts = set(df_train["text"])
leak_valid = df_valid["text"].apply(lambda t: t in train_texts).sum()
leak_test = df_test["text"].apply(lambda t: t in train_texts).sum()
print(f"Jumlah teks validasi yang muncul di data latih: {leak_valid}")
print(f"Jumlah teks uji (test) yang muncul di data latih: {leak_test}")

print("\n[RINGKASAN] Seluruh data SmSA siap digunakan untuk tahap pemodelan Transformer!")

Jumlah teks duplikat di data latih: 67
Jumlah teks dengan label berkontradiksi di data latih: 0
Jumlah teks validasi yang muncul di data latih: 14
Jumlah teks uji (test) yang muncul di data latih: 0

[RINGKASAN] Seluruh data SmSA siap digunakan untuk tahap pemodelan Transformer!
